# CNN. Part 2

> Проверяем подходы, которые обсуждали на лекции. \
> Смотрим предобученные модели

materials inspired by [course](http://wiki.cs.hse.ru/%D0%93%D0%BB%D1%83%D0%B1%D0%B8%D0%BD%D0%BD%D0%BE%D0%B5_%D0%BE%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_1_24/25)

### Нормализация входных данных

Хорошая практика в глубинном обучении - нормализовать входные данные.

In [ ]:
import torch
import numpy as np

from torchvision.datasets import MNIST
import torchvision.transforms.v2 as T

In [ ]:
mnist_train = MNIST('../datasets/mnist', transform=T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]), download=True)

In [ ]:
mnist_train.data.shape

In [ ]:
mnist_train.data

In [ ]:
mnist_train.data.max()

In [ ]:
mean = mnist_train.data.to(torch.float32).mean().item() / 255

mean

In [ ]:
std = mnist_train.data.to(torch.float32).std().item() / 255

std

In [ ]:
normalize = T.Normalize(mean=(mean,), std=(std,))

In [ ]:
mnist_train[0][0]

In [ ]:
normalized_image = normalize(mnist_train[0][0])

In [ ]:
normalized_image

In [ ]:
normalized_image.mean()

In [ ]:
normalized_image.std()

In [ ]:
normalize(mnist_train.data.to(torch.float32) / 255).mean()

In [ ]:
normalize(mnist_train.data.to(torch.float32) / 255).std()

## Аугментация данных

https://pytorch.org/vision/stable/transforms.html

https://pytorch.org/vision/stable/auto_examples/plot_transforms.html#sphx-glr-auto-examples-plot-transforms-py

## Практическая часть

In [ ]:
from torchvision.datasets import CIFAR10

In [ ]:
dataset_train = CIFAR10('../datasets/cifar', train=True, download=True)

In [ ]:
dataset_train.class_to_idx

In [ ]:
dataset_train[0][0]

In [ ]:
dataset_train[0][0]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.imshow(dataset_train[0][0])
plt.show()

In [ ]:
img_matrix = np.array(dataset_train[0][0]) / 255
img_matrix

In [ ]:
dataset_train = CIFAR10('../datasets/cifar', train=True, transform=T.ToTensor())
dataset_train.data.shape

In [ ]:
means = (dataset_train.data / 255).mean(axis=(0, 1, 2))
stds = (dataset_train.data / 255).std(axis=(0, 1, 2))
print(means, stds)
means

In [ ]:
import torchvision.transforms as T

transforms = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

In [ ]:
from torch.utils.data import DataLoader

train_dataset = CIFAR10('../datasets/cifar', train=True, transform=transforms)
valid_dataset = CIFAR10('../datasets/cifar', train=False, transform=transforms)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
import torch.nn as nn
from torch.optim import Optimizer
from tqdm import tqdm


def train(model: nn.Module, data_loader: DataLoader, optimizer: Optimizer, loss_fn, device: torch.device):
    model.train()

    total_loss = 0
    total_correct = 0

    for x, y in tqdm(data_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        output = model(x)

        loss = loss_fn(output, y)

        loss.backward()

        total_loss += loss.item()

        total_correct += (output.argmax(dim=1) == y).sum().item()

        optimizer.step()

    return total_loss / len(data_loader), total_correct / len(data_loader.dataset)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader


@torch.inference_mode()
def evaluate(model: nn.Module, data_loader: DataLoader, loss_fn, device: torch.device):
    model.eval()

    total_loss = 0
    total_correct = 0

    for x, y in tqdm(data_loader):
        x, y = x.to(device), y.to(device)

        output = model(x)

        loss = loss_fn(output, y)

        total_loss += loss.item()

        total_correct += (output.argmax(dim=1) == y).sum().item()

    return total_loss / len(data_loader), total_correct / len(data_loader.dataset)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


sns.set(style='darkgrid')


def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    train_accuracy: list[float],
    valid_accuracy: list[float],
    title: str
):
    plt.figure(figsize=(8, 4))

    plt.title(title + ' loss')

    plt.plot(train_loss, label='Train loss')
    plt.plot(valid_loss, label='Valid loss')
    plt.legend()

    plt.show()

    plt.figure(figsize=(8, 4))

    plt.title(title + ' accuracy')

    plt.plot(train_accuracy, label='Train accuracy')
    plt.plot(valid_accuracy, label='Valid accuracy')
    plt.legend()

    plt.show()

In [ ]:
from IPython.display import clear_output


def fit(model, train_loader, valid_loader, optimizer, loss_fn, device, num_epochs, title):
    train_loss_history, valid_loss_history = [], []
    train_accuracy_history, valid_accuracy_history = [], []

    for epoch in range(num_epochs):
        train_loss, train_accuracy = train(model, train_loader, optimizer, loss_fn, device)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        clear_output()

        plot_stats(
            train_loss_history, valid_loss_history,
            train_accuracy_history, valid_accuracy_history,
            title
        )

        torch.save(model.state_dict(), f"./model_{epoch}.pt")
        torch.save(optimizer.state_dict(), f"./optimizer_{epoch}.pt")

### Эксперименты с архитектурами моделей

In [ ]:
from torch import nn


class FirstModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            # YOUR CODE HERE
            nn.Linear(...),

            nn.ReLU(),
            nn.Linear(1024, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(device)
print(torch.cuda.get_device_name())

loss_fn = nn.CrossEntropyLoss()

In [ ]:
from torch.optim import Adam


# YOUR CODE HERE
model_name = ...
epoch = ...

model = FirstModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)
fit(model, train_loader, valid_loader, optimizer, loss_fn, device, 10, model_name)

In [ ]:
model, optimizer

In [ ]:
# Load model for checkpoints

model = FirstModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

model.load_state_dict(torch.load(f"./model_{epoch}.pt"))
optimizer.load_state_dict(torch.load(f"./optimizer_{epoch}.pt"))


In [ ]:
# Continue training

fit(model, train_loader, valid_loader, optimizer, loss_fn, device, 3, model_name)

In [ ]:
# Add more conv blocks

class SecondModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            # YOUR CODE HERE
            nn.Linear(..., 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
# YOUR CODE HERE
model_name = ...
epoch = ...

model = SecondModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

fit(model, train_loader, valid_loader, optimizer, loss_fn, device, epoch, model_name)

In [1]:
# Add BatchNorm

class ThirdModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            # YOUR CODE HERE
            # add BatchNorm in right place in each block
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Flatten(),

            # YOUR CODE HERE
            nn.Linear(..., 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.net(x)

NameError: name 'nn' is not defined

In [ ]:
# YOUR CODE HERE
model_name = ...
epoch = ...

model = ThirdModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

fit(model, train_loader, valid_loader, optimizer, loss_fn, device, epoch, model_name)

In [ ]:
# Use the previous implementation with additions - add Dropout (choose p at your discretion)

class FourthModel(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, x):
        return self.net(x)

In [ ]:
# YOUR CODE HERE
model_name = ...
epoch = ...

model = FourthModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

fit(model, train_loader, valid_loader, optimizer, loss_fn, device, epoch, model_name)

In [ ]:
train_transforms = T.Compose(
    [
        T.RandomResizedCrop(size=32, scale=(0.8, 1.1)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAdjustSharpness(sharpness_factor=2),
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

test_transforms = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

train_dataset = CIFAR10('../datasets/cifar', train=True, transform=train_transforms)
valid_dataset = CIFAR10('../datasets/cifar', train=False, transform=test_transforms)

train_loader_augs = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
valid_loader_augs = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
?? train_dataset

In [ ]:
# YOUR CODE HERE
model_name = ...
epoch = ... # should we use more then in previous?

model = FourthModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

fit(
    model,
    train_loader_augs, valid_loader_augs,
    optimizer, loss_fn, device, epoch, model_name
)

In [ ]:
train_transforms = T.Compose(
    [
        T.RandAugment(),
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

test_transforms = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

train_dataset = CIFAR10('../datasets/cifar', train=True, transform=train_transforms)
valid_dataset = CIFAR10('../datasets/cifar', train=False, transform=test_transforms)

train_loader_augs_random = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
valid_loader_augs_random = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
model_name = ...
epoch = ...

model = FourthModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

fit(
    model,
    train_loader_augs_random, valid_loader_augs_random,
    optimizer, loss_fn, device, epoch, model_name
)

In [ ]:
train_transforms = T.Compose(
    [
        T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

test_transforms = T.Compose(
    [
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

train_dataset = CIFAR10('cifar', train=True, transform=train_transforms)
valid_dataset = CIFAR10('cifar', train=False, transform=test_transforms)

train_loader_augs_auto = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
valid_loader_augs_auto = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
model_name = ...
epoch = ...

model = FourthModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

fit(
    model,
    train_loader_augs, valid_loader_augs,
    optimizer, loss_fn, device, epoch, model_name
)

In [ ]:
# https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate

def fit(model, train_loader, valid_loader, optimizer, loss_fn, device, num_epochs, title, scheduler=None):
    train_loss_history, valid_loss_history = [], []
    train_accuracy_history, valid_accuracy_history = [], []

    for epoch in range(num_epochs):
        train_loss, train_accuracy = train(model, train_loader, optimizer, loss_fn, device)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn, device)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        train_accuracy_history.append(train_accuracy)
        valid_accuracy_history.append(valid_accuracy)

        clear_output()

        plot_stats(
            train_loss_history, valid_loss_history,
            train_accuracy_history, valid_accuracy_history,
            title
        )

        # added scheduler
        if scheduler is not None:
            scheduler.step()

In [ ]:
from torch.optim.lr_scheduler import StepLR

model_name = ...
epoch = ... # should we use more again?


model = FourthModel().to(device)
optimizer = Adam(model.parameters(), lr=1e-3)
scheduler = StepLR(optimizer, step_size=25, gamma=0.1)


fit(
    model,
    train_loader_augs, valid_loader_augs,
    optimizer, loss_fn, device, epoch, model_name, scheduler
)

**Bonus task:**
- Продолжайте эксперименты для достижения большей точности:
    1. Добавьте больше слоёв и используйте skip connections
    2. ...

### Инференс модели

In [ ]:
@torch.inference_mode()
def predict(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()

    prediction = []

    for x, _ in tqdm(loader):
        output = model(x.to(device)).cpu()

        prediction.append(torch.argmax(output, dim=1))

    prediction = torch.cat(prediction)

    return prediction


In [ ]:
def get_labels(loader):
    labels = []

    for _, y in tqdm(loader):
        labels.append(y)

    return torch.cat(labels, dim=0)

In [ ]:
prediction = predict(model, valid_loader, device)
labels = get_labels(valid_loader)

In [ ]:
prediction

In [ ]:
labels

In [ ]:
torch.mean((prediction == labels).to(torch.float32))

In [ ]:
@torch.inference_mode()
def predict_tta(model: nn.Module, loader: DataLoader, device: torch.device, iterations: int=2):
    model.eval()

    prediction = []

    for iteration in range(iterations):
        single_prediction = []

        for x, _ in tqdm(loader):
            output = model(x.to(device)).cpu()

            single_prediction.append(output)

        prediction.append(torch.vstack(single_prediction))

    prediction = torch.argmax(torch.mean(torch.stack(prediction), dim=0), dim=1)

    return prediction

In [ ]:
transforms = T.Compose(
    [
        T.RandomResizedCrop(size=32, scale=(0.8, 1.2)),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=means, std=stds)
    ]
)

valid_dataset = CIFAR10('../datasets/cifar', train=False, transform=transforms)

valid_loader_augs_tta = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
prediction_tta = predict_tta(model, valid_loader_augs_tta, device, iterations=20)

In [ ]:
torch.mean((prediction_tta == labels).to(torch.float32))

### Предобученные модели

In [ ]:
# https://pytorch.org/vision/stable/models.html
from torchvision.models import alexnet
from torchvision.models import vgg11_bn
from torchvision.models import googlenet
from torchvision.models import resnet18

In [ ]:
model = alexnet()
model

In [ ]:
sum(p.numel() for p in model.classifier.parameters())

In [ ]:
model = vgg11_bn()
model

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
model = googlenet(init_weights=False)
model

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
model = resnet18()
model

In [ ]:
sum(p.numel() for p in model.parameters())

In [ ]:
from torchvision.datasets import OxfordIIITPet

dataset = OxfordIIITPet('../datasets/pets', download=True)

In [ ]:
dataset[0]

In [ ]:
dataset[0][0]

In [ ]:
len(dataset)

In [ ]:
dataset.classes

In [ ]:
num_classes = len(dataset.classes)
num_classes

In [ ]:
transform = T.Compose(
    [
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]
)

train_dataset = OxfordIIITPet('../datasets/pets', transform=transform)
valid_dataset = OxfordIIITPet('../datasets/pets', transform=transform, split='test')

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=8, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

In [ ]:
from torchvision.models import AlexNet_Weights, ResNet18_Weights

In [ ]:
model = alexnet()

In [ ]:
next(iter(model.parameters()))

In [ ]:
model = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)

model

In [ ]:
model.classifier[6] = nn.Linear(in_features=4096, out_features=num_classes)

model

In [ ]:
sum(p.numel() for p in model.features.parameters())

In [ ]:
sum(p.numel() for p in model.classifier.parameters())

In [ ]:
device = torch.device("cuda:0")

In [ ]:
from torch.optim import Adam

loss_fn = torch.nn.CrossEntropyLoss()

model = model.to(device)

model.features.requires_grad_(False)
model.features[8].requires_grad_(True)
model.features[10].requires_grad_(True)

optimizer = Adam([p for p in model.parameters() if p.requires_grad], lr=1e-4)

fit(
    model,
    train_loader, valid_loader,
    optimizer, loss_fn, device, 5, 'Alexnet pretrained'
)

In [ ]:
model = alexnet(weights=AlexNet_Weights.IMAGENET1K_V1)
model.requires_grad_(False)
model.classifier[6] = nn.Linear(in_features=4096, out_features=num_classes)

model = model.to(device)

In [ ]:
optimizer = Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)

fit(
    model,
    train_loader, valid_loader,
    optimizer, loss_fn, device, 5, 'Alexnet not pretrained'
)

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

model

In [ ]:
model.fc = nn.Linear(in_features=512, out_features=num_classes)

model

In [ ]:
model = model.to(device)

model.requires_grad_(False)
model.layer4.requires_grad_(True)
model.fc.requires_grad_(True)

# optimizer = Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3)

optimizer = Adam(
    [
        {'params': model.fc.parameters(), 'lr': 1e-3},
        {'params': model.layer4.parameters(), 'lr': 1e-4}
    ]
)

fit(
    model,
    train_loader, valid_loader,
    optimizer, loss_fn, device, 10, 'ResNet18 pretrained'
)

In [ ]:
model = resnet18()

model.fc = nn.Linear(in_features=512, out_features=num_classes)

model

In [ ]:
model = model.to(device)


optimizer = Adam(model.parameters(), lr=1e-3)

fit(
    model,
    train_loader, valid_loader,
    optimizer, loss_fn, device, 10, 'ResNet18 not pretrained'
)

In [ ]:
torch.save(model.state_dict(), 'resnet.pt')

In [ ]:
model = resnet18(pretrained=False)

In [ ]:
model.load_state_dict(torch.load('resnet.pt'))

In [ ]:
torch.save(
    {
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict()
    },
    'checkpoint.pt'
)